In [55]:
# Import dependencies
import numpy as np
import pandas as pd

In [56]:
# Load in the dataset
df = pd.read_csv('../../datasets/heart.csv')

X = df.drop(columns=['target']).to_numpy()
y = df['target'].to_numpy().reshape((303,1)) # Force the 1 dimension

# Perform Z-score normalization on x
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

In [57]:
# Feature scaling: get quadratic terms

def get_quadratic_terms(X):
    n = X.shape[1]

    # Multiply every column by every other column using broadcasting
    # yields [samples, features, features]
    interactions = X[:, :, np.newaxis] * X[:, np.newaxis, :]

    # Get only unique pairs (upper triangular matrix including diagonal for quadratic)
    rows, cols = np.triu_indices(n, k=0)

    # Extract those features
    return interactions[:, rows, cols]

quadratic_terms = get_quadratic_terms(X)
X = np.hstack((X, quadratic_terms))

In [58]:
# Split into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, shuffle=True)

In [59]:
# Binary Cross Entropy (Log Loss) implementation
def BCELoss(y: np.ndarray, y_hat: np.ndarray):
    # Clip y_hat to be between epsilon and 1-epsilon
    # This prevents log(0)
    y_hat = np.clip(y_hat, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

In [ ]:
# Hyperparameters
learning_rate = 0.0005
epochs = 10000

In [61]:
class LogisticRegression:
    def __init__(self, num_features, learning_rate):
        self.lr = learning_rate
        self.w = np.zeros((num_features, 1))
        self.b = 0

    def forward(self, X: np.ndarray):
        # Get linear component (z) and add a quadratic component (feature scaling)
        z = (X @ self.w) + self.b

        # Apply sigmoid
        return 1 / (1 + np.exp(-z))

    def update_weights(self, X: np.ndarray, y: np.ndarray):
        n = len(y) # Get length of examples

        # Get predictions
        y_hat = self.forward(X)

        # Perform gradient descent for each feature
        dw = (1/n) * (X.T @ (y_hat - y))
        db = (1/n) * np.sum(y_hat - y)

        self.w -= self.lr * dw
        self.b -= self.lr * db

    # Fit the data onto the logistic regression function
    def fit(self, X: np.ndarray, y: np.ndarray, epochs=1000, verbose=False):
        for i in range(epochs):
            # Perform gradient descent
            self.update_weights(X, y)

            # Report loss if verbose
            if verbose and i % 100 == 0:
                loss = BCELoss(y, self.forward(X))
                print(f'[{i}]: Loss: {loss:.4f}')


In [62]:
model = LogisticRegression(X.shape[1], learning_rate=learning_rate)
model.fit(X_train, y_train, epochs, True)

[0]: Loss: 0.6931
[100]: Loss: 0.6876
[200]: Loss: 0.6822
[300]: Loss: 0.6769
[400]: Loss: 0.6717
[500]: Loss: 0.6667
[600]: Loss: 0.6618
[700]: Loss: 0.6570
[800]: Loss: 0.6524
[900]: Loss: 0.6478
[1000]: Loss: 0.6433
[1100]: Loss: 0.6389
[1200]: Loss: 0.6347
[1300]: Loss: 0.6305
[1400]: Loss: 0.6264
[1500]: Loss: 0.6224
[1600]: Loss: 0.6184
[1700]: Loss: 0.6146
[1800]: Loss: 0.6108
[1900]: Loss: 0.6071
[2000]: Loss: 0.6035
[2100]: Loss: 0.5999
[2200]: Loss: 0.5964
[2300]: Loss: 0.5930
[2400]: Loss: 0.5896
[2500]: Loss: 0.5863
[2600]: Loss: 0.5831
[2700]: Loss: 0.5799
[2800]: Loss: 0.5768
[2900]: Loss: 0.5737
[3000]: Loss: 0.5707
[3100]: Loss: 0.5678
[3200]: Loss: 0.5649
[3300]: Loss: 0.5620
[3400]: Loss: 0.5592
[3500]: Loss: 0.5564
[3600]: Loss: 0.5537
[3700]: Loss: 0.5511
[3800]: Loss: 0.5484
[3900]: Loss: 0.5459
[4000]: Loss: 0.5433
[4100]: Loss: 0.5408
[4200]: Loss: 0.5384
[4300]: Loss: 0.5359
[4400]: Loss: 0.5336
[4500]: Loss: 0.5312
[4600]: Loss: 0.5289
[4700]: Loss: 0.5266
[480

In [63]:
# Evaluate model
def eval(X: np.ndarray, y: np.ndarray, model: LogisticRegression):
    # Get predictions (from actual labels)
    y_hat = model.forward(X)
    y_hat = (y_hat >= 0.5).astype(int)

    # Calculate accruacy
    accuracy = np.mean(y_hat == y)
    print(f'Test accuracy: {accuracy:.3f}')

eval(X_test, y_test, model)

Test accuracy: 0.870
